# 01 — JSON-RPC 2.0: The Language MCP Speaks

**Goal:** Understand JSON-RPC 2.0 — the message format that MCP is built on.

## Why JSON-RPC?

MCP needs a standard way to say:
- "Hey server, list your tools" (request)
- "Here are my tools" (response)
- "I'm ready" (notification — no response needed)
- "That failed" (error)

JSON-RPC 2.0 is a simple spec that does exactly this. It's just JSON with a few required fields.

## The three message types

```
REQUEST:       {"jsonrpc": "2.0", "id": 1, "method": "add", "params": {"a": 1, "b": 2}}
RESPONSE:      {"jsonrpc": "2.0", "id": 1, "result": 3}
ERROR:         {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
NOTIFICATION:  {"jsonrpc": "2.0", "method": "ping"}     ← NO id field = no response expected
```

**Key rule:** If a message has an `id`, the other side MUST respond. If no `id`, it's fire-and-forget.

## Exercise 1.1 — Parse JSON-RPC messages

Write a function that takes a JSON string and classifies it as `request`, `notification`, `response`, or `error`.

In [ ]:
import json

def classify_message(raw: str) -> str:
    """Classify a JSON-RPC 2.0 message.
    Returns: 'request', 'notification', 'response', or 'error'
    """
    msg = json.loads(raw)
    
    # TODO: implement this
    # Hints:
    # - Has 'method' field? It's either a request or notification
    # - Has 'id' field? It expects a response (request) vs fire-and-forget (notification)
    # - Has 'result' field? It's a successful response
    # - Has 'error' field? It's an error response
    pass

# Test it
tests = [
    '{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}',
    '{"jsonrpc": "2.0", "method": "notifications/initialized"}',
    '{"jsonrpc": "2.0", "id": 1, "result": {"tools": []}}',
    '{"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Not found"}}',
]
expected = ['request', 'notification', 'response', 'error']

for raw, exp in zip(tests, expected):
    result = classify_message(raw)
    status = '✓' if result == exp else '✗'
    print(f"  {status} {exp}: {result}")

## Exercise 1.2 — Build JSON-RPC messages

Write helper functions to construct valid JSON-RPC messages. These will be your building blocks.

In [ ]:
class JsonRpcBuilder:
    """Builds JSON-RPC 2.0 messages."""
    
    def __init__(self):
        self._next_id = 1
    
    def request(self, method: str, params: dict | None = None) -> dict:
        """Build a request (has id → expects response)."""
        # TODO: return a dict with jsonrpc, id, method, and optionally params
        # Auto-increment self._next_id
        pass
    
    def notification(self, method: str, params: dict | None = None) -> dict:
        """Build a notification (no id → no response expected)."""
        # TODO: return a dict with jsonrpc, method, and optionally params
        # NOTE: NO id field!
        pass
    
    def response(self, id: int, result: any) -> dict:
        """Build a success response."""
        # TODO: return a dict with jsonrpc, id, result
        pass
    
    def error(self, id: int, code: int, message: str) -> dict:
        """Build an error response."""
        # TODO: return a dict with jsonrpc, id, error: {code, message}
        pass


# Test it
rpc = JsonRpcBuilder()

req = rpc.request("tools/list")
print("Request:", json.dumps(req))
assert req["jsonrpc"] == "2.0"
assert req["id"] == 1
assert req["method"] == "tools/list"

req2 = rpc.request("tools/call", {"name": "add", "arguments": {"a": 1, "b": 2}})
print("Request:", json.dumps(req2))
assert req2["id"] == 2  # auto-incremented
assert req2["params"]["name"] == "add"

notif = rpc.notification("notifications/initialized")
print("Notification:", json.dumps(notif))
assert "id" not in notif  # crucial!

resp = rpc.response(1, {"tools": []})
print("Response:", json.dumps(resp))
assert resp["id"] == 1

err = rpc.error(1, -32601, "Method not found")
print("Error:", json.dumps(err))
assert err["error"]["code"] == -32601

print("\n✓ All tests passed!")

## Exercise 1.3 — Build a JSON-RPC dispatcher

A dispatcher receives a request and routes it to the right handler function. This is the core of any RPC server.

Think of it like a Python `match` statement: incoming method name → call the right function.

In [ ]:
class Dispatcher:
    """Routes JSON-RPC requests to handler functions."""
    
    def __init__(self):
        self._handlers: dict[str, callable] = {}
    
    def register(self, method: str, handler: callable):
        """Register a handler for a method name."""
        self._handlers[method] = handler
    
    def dispatch(self, message: dict) -> dict | None:
        """Process a JSON-RPC message. Returns a response dict, or None for notifications."""
        # TODO: implement this
        # 1. If it's a notification (no 'id'), call handler (if registered) and return None
        # 2. If it's a request (has 'id'):
        #    a. Look up the handler by 'method'
        #    b. If not found, return an error response (code -32601)
        #    c. If found, call handler(params) and return a success response
        #    d. If handler raises an exception, return an error response (code -32603)
        pass


# Test it
dispatcher = Dispatcher()

# Register some handlers
def handle_add(params):
    return params["a"] + params["b"]

def handle_greet(params):
    return f"Hello, {params['name']}!"

def handle_fail(params):
    raise ValueError("Something went wrong")

dispatcher.register("add", handle_add)
dispatcher.register("greet", handle_greet)
dispatcher.register("fail", handle_fail)

# Test: successful request
resp = dispatcher.dispatch({"jsonrpc": "2.0", "id": 1, "method": "add", "params": {"a": 3, "b": 4}})
print("add:", resp)
assert resp["result"] == 7

# Test: method not found
resp = dispatcher.dispatch({"jsonrpc": "2.0", "id": 2, "method": "unknown"})
print("unknown:", resp)
assert resp["error"]["code"] == -32601

# Test: handler exception
resp = dispatcher.dispatch({"jsonrpc": "2.0", "id": 3, "method": "fail", "params": {}})
print("fail:", resp)
assert resp["error"]["code"] == -32603

# Test: notification (no response)
result = dispatcher.dispatch({"jsonrpc": "2.0", "method": "greet", "params": {"name": "World"}})
print("notification:", result)
assert result is None  # notifications don't get responses

print("\n✓ All tests passed!")

### Question 1.3
Why does JSON-RPC separate errors into two categories?
- **Protocol errors** (code -32601 "Method not found"): the request itself is invalid
- **Application errors** (code -32603 "Internal error"): the handler crashed

In MCP specifically, there's a third category: **tool errors** (the tool ran but failed). These are NOT JSON-RPC errors — they're returned as a successful response with `isError: true` in the result. Why?

*Your answer:*


## Summary

You now have the building blocks:

| Component | What it does |
|-----------|--------------|
| `classify_message()` | Tells you what kind of message you received |
| `JsonRpcBuilder` | Creates well-formed requests, responses, notifications, errors |
| `Dispatcher` | Routes requests to the right handler, wraps results/errors |

**Next notebook:** Transport layer — sending these messages over stdin/stdout